In [19]:
import os
import sys
import time
import pickle
import pandas as pd
import numpy as np
from datetime import datetime

# =============================================================================
# SHARED CONFIGURATION PARAMETERS
# =============================================================================

# -- File & Path Configuration --
HDF_FILE_PATH = '../../data/raw/all_hourly_data.h5'
FEATURE_CLASSIFICATION_PATH = '../../data/processed/eda_results_corrected/feature_classification.csv'
OUTPUT_DIR = 'phase_1_outputs'

# -- Experiment Configuration --
DRY_RUN = False  # Set to False for full dataset
DRY_RUN_PATIENTS = 1000  # Number of patients for dry run
USE_CACHED_PREPROCESSING = True  # Use cache if available

# -- Feature Engineering Options --
CALCULATE_TRENDS = True  # Include trend calculations
WINDOW_SIZE = 24  # Hours of data for feature extraction
GAP_TIME = 6      # Hours of gap before prediction

# -- Target and Reproducibility --
TARGET_VARIABLE = 'mort_hosp'
SEED = 42

# -- Hyperparameter Tuning Configuration --
N_OPTUNA_TRIALS = 15  # Number of trials for hyperparameter search
OPTUNA_TIMEOUT = 1800 # Timeout in seconds

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [20]:
print("=" * 70)
print("PHASE 1: CLEAN DATA PREPROCESSING")
print("=" * 70)

preprocessing_start = time.time()

# Import and run preprocessing
try:
    # Import the clean preprocessing script as a module
    import importlib.util
    spec = importlib.util.spec_from_file_location("data_preprocessing_clean", "data_preprocessing_clean.py")
    if spec is None:
        raise ImportError("Could not load data_preprocessing_clean.py")
    
    preprocessing_module = importlib.util.module_from_spec(spec)
    if spec.loader is None:
        raise ImportError("No loader found for data_preprocessing_clean.py")
    
    # Execute the preprocessing script
    print(f"Starting clean data preprocessing at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    spec.loader.exec_module(preprocessing_module)
    
    # Create configuration dictionary from notebook parameters
    config = {
        'HDF_FILE_PATH': HDF_FILE_PATH,
        'FEATURE_CLASSIFICATION_PATH': FEATURE_CLASSIFICATION_PATH,
        'OUTPUT_DIR': OUTPUT_DIR,
        'DRY_RUN': DRY_RUN,
        'DRY_RUN_PATIENTS': DRY_RUN_PATIENTS,
        'USE_CACHED_PREPROCESSING': USE_CACHED_PREPROCESSING,
        'CALCULATE_TRENDS': CALCULATE_TRENDS,
        'WINDOW_SIZE': WINDOW_SIZE,
        'GAP_TIME': GAP_TIME,
        'TARGET_VARIABLE': TARGET_VARIABLE,
        'SEED': SEED
    }
    
    # Call the main function explicitly with configuration
    preprocessing_module.main(config)
    
    preprocessing_time = time.time() - preprocessing_start
    print(f"\n✓ Clean data preprocessing completed in {preprocessing_time/60:.2f} minutes")
    
except Exception as e:
    print(f"❌ Error during preprocessing: {e}")
    raise


2025-07-02 10:25:56,415 [INFO] - Loading data...


PHASE 1: CLEAN DATA PREPROCESSING
Starting clean data preprocessing at 2025-07-02 10:25:56


2025-07-02 10:26:00,925 [INFO] - Age filtered: 32550 patients, 2080959 records
2025-07-02 10:26:07,491 [INFO] - Time window filtered: 22591 patients, (542184, 104) time-series
2025-07-02 10:26:51,198 [INFO] - Demographics processed: (22591, 4)
2025-07-02 10:26:51,199 [INFO] - Demographic encoders: ['gender_encoded', 'ethnicity_encoded', 'insurance_encoded']
2025-07-02 10:26:51,220 [INFO] - Splits: train=14825, val=2118, test=5648 subjects
2025-07-02 10:26:51,591 [INFO] - Engineering features for 6 categories
2025-07-02 10:33:00,561 [INFO] - Features engineered: (14825, 454)
2025-07-02 10:33:00,726 [INFO] - Engineering features for 6 categories
2025-07-02 10:33:53,553 [INFO] - Features engineered: (2118, 454)
2025-07-02 10:33:53,742 [INFO] - Engineering features for 6 categories
2025-07-02 10:36:17,337 [INFO] - Features engineered: (5648, 454)
2025-07-02 10:36:17,396 [INFO] - ✓ All features are numeric - no categorical encoding needed
2025-07-02 10:36:19,404 [INFO] - ✓ Outlier handling:


✓ Clean data preprocessing completed in 10.41 minutes


## 3. XGBoost Model Training & Evaluation

Run XGBoost analysis with hyperparameter optimization using the preprocessed data:


In [21]:
print("=" * 70)
print("PHASE 2: XGBOOST MODEL TRAINING & EVALUATION")
print("=" * 70)

xgboost_start = time.time()

# Initialize variables for results capture
auroc_results = None
auprc_results = None
final_model = None
xgboost_results = None

try:
    # Import and run XGBoost analysis script  
    import importlib.util
    spec = importlib.util.spec_from_file_location("xgboost_analysis", "xgboost_analysis.py")
    if spec is None:
        raise ImportError("Could not load xgboost_analysis.py")
    
    xgboost_module = importlib.util.module_from_spec(spec)
    if spec.loader is None:
        raise ImportError("No loader found for xgboost_analysis.py")
    
    print(f"Starting XGBoost analysis at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    # Execute the XGBoost analysis module and call main function
    spec.loader.exec_module(xgboost_module)
    
    # Create configuration dictionary from notebook parameters
    config = {
        'OUTPUT_DIR': OUTPUT_DIR,
        'DRY_RUN': DRY_RUN,
        'DRY_RUN_PATIENTS': DRY_RUN_PATIENTS,
        'CALCULATE_TRENDS': CALCULATE_TRENDS,
        'WINDOW_SIZE': WINDOW_SIZE,
        'GAP_TIME': GAP_TIME,
        'TARGET_VARIABLE': TARGET_VARIABLE,
        'SEED': SEED,
        'N_OPTUNA_TRIALS': N_OPTUNA_TRIALS,
        'OPTUNA_TIMEOUT': OPTUNA_TIMEOUT
    }
    
    # Call the main function explicitly with configuration
    xgboost_module.main(config)
    
    # Load the results that were saved by the XGBoost script
    results_path = os.path.join(OUTPUT_DIR, 'results_xgboost_baseline.pkl')
    if os.path.exists(results_path):
        with open(results_path, 'rb') as f:
            xgboost_results = pickle.load(f)
        
        # Extract key metrics for display
        auroc_results = {
            'point_estimate': xgboost_results['test_auroc'],
            'ci_lower': xgboost_results['test_auroc_ci_lower'],
            'ci_upper': xgboost_results['test_auroc_ci_upper']
        }
        
        auprc_results = {
            'point_estimate': xgboost_results['test_auprc'], 
            'ci_lower': xgboost_results['test_auprc_ci_lower'],
            'ci_upper': xgboost_results['test_auprc_ci_upper']
        }
        
        print(f"\n✅ XGBoost Results Successfully Captured:")
        print(f"   AUROC: {auroc_results['point_estimate']:.4f} "
              f"(95% CI: {auroc_results['ci_lower']:.4f}-{auroc_results['ci_upper']:.4f})")
        print(f"   AUPRC: {auprc_results['point_estimate']:.4f} "
              f"(95% CI: {auprc_results['ci_lower']:.4f}-{auprc_results['ci_upper']:.4f})")
    else:
        print("⚠️  Results file not found - metrics may not be available in summary")
    
    xgboost_time = time.time() - xgboost_start
    print(f"\n✓ XGBoost analysis completed in {xgboost_time/60:.2f} minutes")
    
except Exception as e:
    print(f"❌ Error during XGBoost analysis: {e}")
    print(f"   This may be due to XGBoost library installation issues")
    print(f"   Preprocessing completed successfully - data is ready for manual analysis")
    xgboost_time = time.time() - xgboost_start
    print(f"   Attempted runtime: {xgboost_time/60:.2f} minutes")
    # Don't raise - allow pipeline to continue and show preprocessing results


2025-07-02 10:36:46,349 [INFO] - Loading preprocessed data with prefix: preprocessed_mort_hosp_trends_True_window_24_gap_6_seed_42
2025-07-02 10:36:46,426 [INFO] - Data shapes: X_train=(14825, 458), X_val=(2118, 458), X_test=(5648, 458)
2025-07-02 10:36:46,426 [INFO] - Using preprocessed data directly - no additional cleaning required
2025-07-02 10:36:46,427 [INFO] - Creating new Optuna study with 15 trials...
[I 2025-07-02 10:36:46,427] A new study created in memory with name: no-name-aa726b37-1f37-4819-ac33-761013d642aa


PHASE 2: XGBOOST MODEL TRAINING & EVALUATION
Starting XGBoost analysis at 2025-07-02 10:36:46


/Users/aaronge/miniconda3/envs/mimic_extract/lib/python3.7/site-packages/xgboost/sklearn.py:797: UserWarning: `eval_metric` in `fit` method is deprecated for better compatibility with scikit-learn, use `eval_metric` in constructor or`set_params` instead.
  UserWarning,
/Users/aaronge/miniconda3/envs/mimic_extract/lib/python3.7/site-packages/xgboost/sklearn.py:797: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  UserWarning,
[I 2025-07-02 10:37:01,368] Trial 0 finished with value: 0.8273089609675646 and parameters: {'n_estimators': 250, 'learning_rate': 0.0026918146370950085, 'max_depth': 4, 'subsample': 0.9372288424124744, 'colsample_bytree': 0.9668803936207525, 'gamma': 0.00010852155940343507, 'min_child_weight': 2}. Best is trial 0 with value: 0.8273089609675646.
/Users/aaronge/miniconda3/envs/mimic_extract/lib/python3.7/site-packages/xgboost/sklearn.py:


Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.97      0.96      5077
           1       0.63      0.46      0.53       571

    accuracy                           0.92      5648
   macro avg       0.79      0.71      0.74      5648
weighted avg       0.91      0.92      0.91      5648


Confusion Matrix:
[[4925  152]
 [ 310  261]]

✅ XGBoost Results Successfully Captured:
   AUROC: 0.9121 (95% CI: 0.9012-0.9228)
   AUPRC: 0.6195 (95% CI: 0.5843-0.6562)

✓ XGBoost analysis completed in 13.02 minutes


## 4. Elastic Net Model Training & Evaluation

Run Elastic Net analysis with hyperparameter optimization using the preprocessed data. Note that Elastic Net requires imputation of missing values since it cannot handle NaN values like XGBoost can:


In [23]:
print("=" * 70)
print("PHASE 3: ELASTIC NET MODEL TRAINING & EVALUATION")
print("=" * 70)

elastic_net_start = time.time()

try:
    # Import and run Elastic Net analysis script  
    import importlib.util
    spec = importlib.util.spec_from_file_location("elastic_net_analysis", "elastic_net_analysis.py")
    if spec is None:
        raise ImportError("Could not load elastic_net_analysis.py")
    
    elastic_net_module = importlib.util.module_from_spec(spec)
    if spec.loader is None:
        raise ImportError("No loader found for elastic_net_analysis.py")
    
    print(f"Starting Elastic Net analysis at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    # Execute the Elastic Net analysis module
    spec.loader.exec_module(elastic_net_module)
    
    # Create configuration dictionary from notebook parameters
    config = {
        'OUTPUT_DIR': OUTPUT_DIR,
        'DRY_RUN': DRY_RUN,
        'DRY_RUN_PATIENTS': DRY_RUN_PATIENTS,
        'CALCULATE_TRENDS': CALCULATE_TRENDS,
        'WINDOW_SIZE': WINDOW_SIZE,
        'GAP_TIME': GAP_TIME,
        'TARGET_VARIABLE': TARGET_VARIABLE,
        'SEED': SEED,
        'N_OPTUNA_TRIALS': N_OPTUNA_TRIALS,
        'OPTUNA_TIMEOUT': OPTUNA_TIMEOUT
    }
    
    # Call the main function
    elastic_net_module.main(config)
    
    elastic_net_time = time.time() - elastic_net_start
    print(f"\\n✓ Elastic Net analysis completed in {elastic_net_time/60:.2f} minutes")
    
except Exception as e:
    print(f"❌ Error during Elastic Net analysis: {e}")
    elastic_net_time = time.time() - elastic_net_start
    print(f"   Attempted runtime: {elastic_net_time/60:.2f} minutes")


2025-07-02 10:55:43,024 [INFO] - Loading preprocessed data with prefix: preprocessed_mort_hosp_trends_True_window_24_gap_6_seed_42


2025-07-02 10:55:43,121 [INFO] - Data shapes: X_train=(14825, 458), X_val=(2118, 458), X_test=(5648, 458)
2025-07-02 10:55:43,122 [INFO] - Using preprocessed data directly - no additional cleaning required
2025-07-02 10:55:43,123 [INFO] - Creating new Optuna study with 15 trials...
[I 2025-07-02 10:55:43,123] A new study created in memory with name: no-name-66df32dd-93a4-47a6-91f6-36862bc1002f


PHASE 3: ELASTIC NET MODEL TRAINING & EVALUATION
Starting Elastic Net analysis at 2025-07-02 10:55:43


[I 2025-07-02 10:57:16,932] Trial 0 finished with value: 0.8563255909840572 and parameters: {'C': 0.2352272775179274, 'l1_ratio': 0.62277687013656}. Best is trial 0 with value: 0.8563255909840572.
[I 2025-07-02 10:59:11,831] Trial 1 finished with value: 0.8516919618314616 and parameters: {'C': 1.1154804501265205, 'l1_ratio': 0.23858982642756132}. Best is trial 0 with value: 0.8563255909840572.
[I 2025-07-02 10:59:15,069] Trial 2 finished with value: 0.8471761368098641 and parameters: {'C': 0.0007893446907770941, 'l1_ratio': 0.407321012677213}. Best is trial 0 with value: 0.8563255909840572.
[I 2025-07-02 10:59:41,083] Trial 3 finished with value: 0.8619286499646588 and parameters: {'C': 0.03089284100404184, 'l1_ratio': 0.3372468692494016}. Best is trial 3 with value: 0.8619286499646588.
[I 2025-07-02 11:01:13,100] Trial 4 finished with value: 0.8523472473101389 and parameters: {'C': 0.38280064339330155, 'l1_ratio': 0.0484259301720007}. Best is trial 3 with value: 0.8619286499646588.
[I


Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.82      0.89      5077
           1       0.34      0.81      0.48       571

    accuracy                           0.82      5648
   macro avg       0.66      0.82      0.69      5648
weighted avg       0.91      0.82      0.85      5648


Confusion Matrix:
[[4187  890]
 [ 107  464]]
\n✓ Elastic Net analysis completed in 22.39 minutes
